# 05 — Bipotenciostato (BiStat) y arreglo multicanal WE32

Este notebook cubre dos capacidades multi-electrodo:

**BiStat** — un segundo canal de electrodo de trabajo (WE2) incorporado en algunos dispositivos Ivium. Tanto WE1 como WE2 comparten el mismo electrodo de referencia y contraelectrodo, pero aplican potenciales independientes y miden corrientes independientes. Común en:
- Experimentos con electrodo de disco-anillo rotante (RRDE)
- Experimentos generador-colector
- Celdas de flujo de doble electrodo

**WE32** — un accesorio multiplexor de arreglo de 32 electrodos de trabajo. Los 32 canales se miden simultáneamente en una llamada a `read_we32_currents()`. Común en:
- Arreglos de detección (electroquímica combinatoria)
- Mapeo de corrosión
- Arreglos de sensores

### Requisitos previos
- Driver abierto, dispositivo conectado (ver `02_device_and_instance_management.ipynb`)
- **Sección BiStat**: el dispositivo debe soportar BiStat (p.ej., IviumStat con opción BiStat)
- **Sección WE32**: el accesorio WE32 debe estar conectado
- Referencia de manejo de errores: `01_getting_started.ipynb` — Sección 4

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
Pyvium.connect_device()
print("Ready")

---

# Parte A — BiStat (Bipotenciostato)

## A1. Modos de conexión BiStat

Usa el modo de conexión 10 u 11 para habilitar el BiStat:

| Código | Modo |
|------|------|
| 10 | BiStat 4EL (4 electrodos, por defecto para RRDE) |
| 11 | BiStat 2EL (2 electrodos) |

WE1 es el electrodo de disco; WE2 es el electrodo de anillo en una configuración RRDE típica.

In [ ]:
Pyvium.set_connection_mode(10)  # BiStat 4EL
print("BiStat 4EL mode selected")

## A2. Modo BiStat y Auto E-ranging

`set_bistat_mode()` controla dos ajustes vinculados simultáneamente:

| Valor | Barrido BiStat | Marco de referencia WE2 | Auto E-ranging |
|-------|----------------|-------------------------|----------------|
| 0 | Estándar (desplazamiento estático WE2) | Potencial WE2 referenciado a **RE** | Desactivado |
| 1 | Barrido (WE2 sigue el barrido WE1) | Potencial WE2 referenciado a **WE1** | Activado |

- **Modo 0 (estándar):** WE2 se mantiene en un desplazamiento fijo desde el electrodo de referencia. Usar para RRDE donde el anillo necesita un potencial absoluto (p.ej., "mantener anillo a +0.4 V vs. Ag/AgCl").
- **Modo 1 (barrido):** WE2 sigue a WE1 con un desplazamiento fijo. Usar para experimentos generador-colector donde quieres que WE2 se mantenga un número fijo de voltios por delante de WE1 durante todo un barrido.

> `connect_device()` siempre restablece el Auto E-ranging a Desactivado (modo 0). Si necesitas el modo de barrido, llama a `set_bistat_mode(1)` explícitamente después de conectar.

In [ ]:
Pyvium.set_bistat_mode(0)  # estándar: potencial WE2 fijo, auto-Eranging desactivado
print("BiStat mode: standard (static WE2 potential)")

## A3. Establecer rangos de corriente para WE1 y WE2

WE2 (el anillo) típicamente lleva una fracción de corriente menor, por lo que puedes usar un rango más fino.

Códigos de rango para WE2 (BiStat):

| Código | Rango |
|------|-------|
| 0 | 10 mA |
| 1 | 1 mA |
| 2 | 100 µA |
| 3 | 10 µA |
| 4 | 1 µA |

In [ ]:
Pyvium.set_current_range(2)      # WE1 (disco): 100 µA
Pyvium.set_we2_current_range(2)  # WE2 (anillo): 100 µA
print("WE1 and WE2 current ranges: 100 µA")

## A4. Establecer potenciales independientes

- `set_potential(E)` — establece el potencial de WE1 (vs. electrodo de referencia)
- `set_we2_potential(E)` — establece el potencial de WE2; el marco de referencia depende del modo BiStat:
  - **Modo 0** (estándar): el valor es un desplazamiento relativo a RE
  - **Modo 1** (barrido): el valor es un desplazamiento relativo a WE1

In [ ]:
WE1_POTENTIAL = 0.0   # V — potencial del disco vs. referencia
WE2_OFFSET    = 0.4   # V — desplazamiento del anillo relativo a WE1

Pyvium.set_potential(WE1_POTENTIAL)
Pyvium.set_we2_potential(WE2_OFFSET)
print(f"WE1 (disk): {WE1_POTENTIAL} V")
print(f"WE2 (ring): WE1 + {WE2_OFFSET} V = {WE1_POTENTIAL + WE2_OFFSET} V vs. ref")

## A5. Medición de doble canal (ejemplo RRDE)

Ambos canales se leen de forma independiente. Este ejemplo simula un mantenimiento estilo RRDE: el disco está a un potencial reductor mientras el anillo se mantiene a un potencial oxidante para detectar el producto.

In [ ]:
DURATION_S = 10.0
INTERVAL_S = 0.2

Pyvium.set_cell_on()
time.sleep(0.1)

timestamps, I_disk, I_ring = [], [], []
t_start = time.time()

while (time.time() - t_start) < DURATION_S:
    t   = time.time() - t_start
    I1  = Pyvium.get_current()      # WE1 (disco)
    I2  = Pyvium.get_we2_current()  # WE2 (anillo)
    timestamps.append(t)
    I_disk.append(I1 * 1e6)   # µA
    I_ring.append(I2 * 1e6)
    time.sleep(INTERVAL_S)

Pyvium.set_cell_off()

# Calcular eficiencia de colección N = -I_ring / I_disk (RRDE ideal)
avg_disk = np.mean(I_disk)
avg_ring = np.mean(I_ring)
if avg_disk != 0:
    N = -avg_ring / avg_disk
    print(f"Ring collection efficiency N ≈ {N:.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(timestamps, I_disk, 'b-', label=f'WE1 disk (mean {avg_disk:.2f} µA)')
ax.plot(timestamps, I_ring, 'r-', label=f'WE2 ring (mean {avg_ring:.2f} µA)')
ax.set_xlabel("Time (s)")
ax.set_ylabel("Current (µA)")
ax.set_title("BiStat — Dual Channel Measurement")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# Parte B — Arreglo multicanal WE32

> 🔬 **Sin verificar — hardware no disponible.** Los métodos WE32 están implementados según la especificación de la DLL de IviumSoft pero no han sido verificados con un accesorio WE32 físico. La API y la estructura del código deberían ser correctas, pero los rangos de parámetros, las restricciones de temporización y los valores de retorno no han sido confirmados en hardware real. Se agradece la retroalimentación de los usuarios de WE32.

El accesorio WE32 conecta 32 electrodos de trabajo a un único potenciostato. Todos los canales comparten el mismo electrodo de referencia y contraelectrodo.

## B1. Seleccionar el canal WE32 activo

`set_we32_channel(index)` selecciona un canal como el electrodo actualmente activo (para métodos y modo directo). Esto **no** afecta a `read_we32_currents()`, que siempre lee todos los 32 simultáneamente.

In [ ]:
Pyvium.set_connection_mode(1)  # volver al EStat estándar para WE32

# Seleccionar canal 1 como electrodo activo
Pyvium.set_we32_channel(1)
print("WE32 channel 1 selected as active")

## B2. Establecer desplazamientos

Los desplazamientos WE32 modifican el potencial aplicado en canales individuales (–2 V a +2 V). Permiten que cada electrodo sea polarizado ligeramente de forma diferente, útil para:
- Compensar variaciones de OCV en el arreglo
- Ejecutar experimentos con gradiente de potencial

**Desplazamiento de un solo canal:**

In [ ]:
# Establecer canal 3 a +0.1 V de desplazamiento
Pyvium.set_we32_offset(3, 0.1)
print("Channel 3 offset: +0.1 V")

# Usar channel_index=0 para aplicar el mismo desplazamiento a TODOS los canales
Pyvium.set_we32_offset(0, 0.0)  # restablecer todos a 0 V
print("All channels reset to 0 V offset")

**Configuración de desplazamiento masivo** — proporcionar una lista de valores para múltiples canales a la vez:

In [ ]:
# Establecer desplazamientos para 8 canales usando un gradiente lineal
N_CHANNELS = 8
offsets = [i * 0.05 for i in range(N_CHANNELS)]  # 0 a 0.35 V en pasos de 50 mV

Pyvium.set_we32_offsets(N_CHANNELS, offsets)
print(f"Applied offsets (V): {[f'{v:.2f}' for v in offsets]}")

## B3. Leer de vuelta los desplazamientos configurados

In [ ]:
readback = Pyvium.get_we32_offsets(N_CHANNELS)
print("Offset readback:")
for i, v in enumerate(readback, start=1):
    print(f"  Channel {i:2d}: {v:+.4f} V")

## B4. Lectura simultánea de corriente en 32 canales

`read_we32_currents()` es la función de medición principal del WE32. Devuelve una lista de 32 valores de corriente, todos muestreados en el mismo instante.

> **Límite del modo simultáneo:** En modo simultáneo WE32 la tasa de muestreo máxima es **10 muestras/s** (intervalo mínimo de 0.1 s entre llamadas). Cada canal puede llevar un máximo de ±1 mA. Superar estos límites producirá lecturas poco fiables. Para adquisición de series temporales, ajusta el ritmo del bucle en consecuencia (ver B6).

In [ ]:
Pyvium.set_potential(0.2)
Pyvium.set_cell_on()
time.sleep(0.1)  # asentamiento

currents = Pyvium.read_we32_currents()

print(f"WE32 currents (32 channels, all in A):")
for i, I in enumerate(currents, start=1):
    print(f"  Ch {i:2d}: {I:.3e} A")

Pyvium.set_cell_off()

## B5. Visualización del mapa de corriente WE32

Un caso de uso común es mostrar las 32 corrientes como un mapa espacial (p.ej., un arreglo de electrodos de 4×8 en un sustrato).

In [ ]:
Pyvium.set_potential(0.2)
Pyvium.set_cell_on()
time.sleep(0.1)

currents = Pyvium.read_we32_currents()
Pyvium.set_cell_off()

currents_uA = np.array(currents) * 1e6  # convertir a µA

# Remodelar como 4 filas × 8 columnas
grid = currents_uA.reshape(4, 8)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(grid, cmap='RdYlGn', aspect='auto')
plt.colorbar(im, ax=ax, label='Current (µA)')

# Anotar cada celda con número de canal
for row in range(4):
    for col in range(8):
        ch = row * 8 + col + 1
        ax.text(col, row, f'{ch}\n{grid[row, col]:.1f}',
                ha='center', va='center', fontsize=8)

ax.set_title("WE32 Current Map (4×8 array)")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.tight_layout()
plt.show()

## B6. Adquisición WE32 resuelta en el tiempo

Llama a `read_we32_currents()` en un bucle para construir una serie temporal en los 32 canales.

> Mantén el intervalo ≥ 0.1 s (máximo 10 muestras/s en modo simultáneo). La sobrecarga del bucle añade algo de tiempo sobre `time.sleep`, así que establece `INTERVAL_S` ligeramente por debajo de tu objetivo para compensarlo.

In [ ]:
DURATION_S = 5.0
INTERVAL_S = 0.5
CHANNELS_TO_SHOW = [0, 7, 15, 31]  # índices de canal basados en 0

Pyvium.set_potential(0.1)
Pyvium.set_cell_on()
time.sleep(0.1)

timeline = []
all_readings = []
t_start = time.time()

while (time.time() - t_start) < DURATION_S:
    t = time.time() - t_start
    snapshot = Pyvium.read_we32_currents()
    timeline.append(t)
    all_readings.append([c * 1e6 for c in snapshot])  # µA
    time.sleep(INTERVAL_S)

Pyvium.set_cell_off()

all_readings = np.array(all_readings)  # forma: (n_puntos_tiempo, 32)

fig, ax = plt.subplots(figsize=(9, 4))
for ch_idx in CHANNELS_TO_SHOW:
    ax.plot(timeline, all_readings[:, ch_idx], label=f'Ch {ch_idx+1}')

ax.set_xlabel("Time (s)")
ax.set_ylabel("Current (µA)")
ax.set_title("WE32 — Selected Channels Over Time")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Limpieza

In [ ]:
Pyvium.set_cell_off()
Pyvium.disconnect_device()
Pyvium.close_driver()
print("Done")

---

## Resumen

### BiStat

| Tarea | Método |
|------|-------|
| Habilitar BiStat | `set_connection_mode(10)` o `(11)` |
| Configurar modo BiStat / auto-Eranging | `set_bistat_mode(0 o 1)` |
| Establecer desplazamiento de potencial WE2 | `set_we2_potential(V)` |
| Establecer rango de corriente WE2 | `set_we2_current_range(n)` |
| Leer corriente WE2 | `get_we2_current()` → float |

### WE32 *(sin verificar — no hay hardware disponible)*

| Tarea | Método | Estado |
|------|--------|-------|
| Seleccionar canal activo | `set_we32_channel(index)` | 🔬 sin verificar |
| Establecer desplazamiento de un canal | `set_we32_offset(index, V)` | 🔬 sin verificar |
| Establecer desplazamientos de múltiples canales | `set_we32_offsets(n, [V, ...])` | 🔬 sin verificar |
| Leer desplazamientos configurados | `get_we32_offsets(n)` → list | 🔬 sin verificar |
| Leer las 32 corrientes | `read_we32_currents()` → list[32] | 🔬 sin verificar |

## Siguiente

- **`06_method_mode.ipynb`** — Ejecutar RRDE, EIS u cualquier otro archivo de método automáticamente